# Demo 08. Euler and Runge-Kutta: Order of Convergence

**Module 5** (explicit initial-value problems), sections 5.1 through 5.4.

A one-step method advances the solution of $y'=f(t,y)$, $y(t_0)=y_0$ by a formula that uses only the current state. The local truncation error is the per-step error; the global error is what accumulates over a full integration. For a method of order $p$ the global error at a fixed final time scales as $Ch^p$. This demo integrates a problem with a known solution using forward Euler, a second-order Runge-Kutta method, and classical RK4, then reads the order off the log-log plot of global error against step size.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Local truncation error and global order

Forward Euler $y_{k+1}=y_k+hf(t_k,y_k)$ has local truncation error $O(h^2)$ per step and global error $O(h)$, so it is first order. A second-order Runge-Kutta method (here the explicit midpoint rule) evaluates the slope once at the interval start and once at the midpoint, giving global error $O(h^2)$. Classical RK4 combines four slope evaluations per step and is globally $O(h^4)$. Reducing $h$ by a factor of two divides the global error by roughly $2^p$, so on log-log axes the error against $h$ is a line of slope $p$.

In [ ]:
# ---------------------------------------------------------------------------
# Three explicit one-step methods, each advancing y' = f(t, y) by one step
# ---------------------------------------------------------------------------
# Every method returns the arrays of times and states over [t0, t_final]. The
# only difference between them is how the step slope is formed: Euler uses the
# slope at the left endpoint, RK2 averages two slopes, RK4 combines four.

def euler_step(f, t, y, h):
    """Forward Euler: single slope at the left endpoint. Order 1."""
    return y + h * f(t, y)

def rk2_step(f, t, y, h):
    """Explicit midpoint rule: slope at the start, then re-sampled at the
    midpoint using the Euler prediction. Order 2."""
    k1 = f(t, y)
    k2 = f(t + 0.5 * h, y + 0.5 * h * k1)
    return y + h * k2

def rk4_step(f, t, y, h):
    """Classical fourth-order Runge-Kutta: a weighted average of four slopes."""
    k1 = f(t, y)
    k2 = f(t + 0.5 * h, y + 0.5 * h * k1)
    k3 = f(t + 0.5 * h, y + 0.5 * h * k2)
    k4 = f(t + h,       y + h * k3)
    return y + h / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

STEPPERS = {"Euler": euler_step, "RK2": rk2_step, "RK4": rk4_step}

def integrate(stepper, f, y0, t0, t_final, h):
    """Advance from t0 to t_final with the given one-step method."""
    n = int(round((t_final - t0) / h))
    ts = np.empty(n + 1); ys = np.empty(n + 1)
    ts[0], ys[0] = t0, y0
    for k in range(n):
        ys[k + 1] = stepper(f, ts[k], ys[k], h)
        ts[k + 1] = ts[k] + h
    return ts, ys

In [ ]:
# ---------------------------------------------------------------------------
# Two test problems with known closed-form solutions
# ---------------------------------------------------------------------------
# A known exact solution lets us measure the true global error. Both problems
# are non-stiff on the chosen interval, so the step size is limited by accuracy
# rather than stability (stability is the subject of demo 15).
PROBLEMS = {
    "y' = y,      y(0)=1      (exact e^t)":
        (lambda t, y: y,              1.0, 0.0, 1.0, lambda t: np.exp(t)),
    "y' = -2 t y, y(0)=1  (exact e^{-t^2})":
        (lambda t, y: -2.0 * t * y,   1.0, 0.0, 2.0, lambda t: np.exp(-t * t)),
}

In [ ]:
# ---------------------------------------------------------------------------
# Global error at the final time vs step size: read the order off the slope
# ---------------------------------------------------------------------------
def show_order(problem=list(PROBLEMS)[0]):
    """For each method, halve h repeatedly and fit the order to the global
    error at t_final."""
    f, y0, t0, tf, exact = PROBLEMS[problem]
    hs = (tf - t0) / (10 * 2 ** np.arange(0, 6))     # h from (tf-t0)/10 down
    y_true = exact(tf)

    plt.figure()
    for name, step in STEPPERS.items():
        err = np.array([abs(integrate(step, f, y0, t0, tf, h)[1][-1] - y_true)
                        for h in hs])
        slope = np.polyfit(np.log(hs), np.log(err + 1e-18), 1)[0]
        plt.loglog(hs, err + 1e-18, ".-", label=f"{name}: order {slope:.2f}")
    plt.xlabel("step size h"); plt.ylabel("global error at final time")
    plt.title("Convergence order of explicit one-step methods")
    plt.legend(); plt.show()
    print("fitted slopes recover the theoretical orders 1 (Euler), 2 (RK2), 4 (RK4)")

show_order()

In [ ]:
# ---------------------------------------------------------------------------
# The solutions themselves at a deliberately coarse step size
# ---------------------------------------------------------------------------
# At a coarse h the accuracy gap is visible directly: Euler drifts off the true
# curve while RK4 stays on it.
def show_solutions(problem=list(PROBLEMS)[0], h=0.25):
    """Overlay each method's numerical solution on the exact curve."""
    f, y0, t0, tf, exact = PROBLEMS[problem]
    tt = np.linspace(t0, tf, 400)
    plt.figure()
    plt.plot(tt, exact(tt), "k-", lw=2, label="exact")
    for name, step in STEPPERS.items():
        ts, ys = integrate(step, f, y0, t0, tf, h)
        plt.plot(ts, ys, ".--", label=f"{name} (h={h})")
    plt.xlabel("t"); plt.ylabel("y"); plt.legend()
    plt.title("Numerical solutions at a coarse step size")
    plt.show()

show_solutions()

In [ ]:
# ---------------------------------------------------------------------------
# Interactive controls
# ---------------------------------------------------------------------------
interact(
    show_order,
    problem=Dropdown(options=list(PROBLEMS), description="problem"),
);
interact(
    show_solutions,
    problem=Dropdown(options=list(PROBLEMS), description="problem"),
    h=FloatSlider(value=0.25, min=0.02, max=0.5, step=0.02, description="step h"),
);

## Summary

- The global error of a one-step method of order $p$ scales as $h^p$, so a log-log plot of error against step size is a line of slope $p$.
- Forward Euler is first order, the explicit midpoint rule is second order, and classical RK4 is fourth order; the fitted slopes recover 1, 2, and 4.
- Higher order buys accuracy per step at the cost of more slope evaluations per step, a tradeoff distinct from the stability limit studied in demo 15.